In [1]:
with open('dataset/input.txt', 'r') as f:
    text = f.read()

In [2]:
import torch

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using: {device}")

Using: cpu


In [3]:
def tokenize(text):
    tokens = []
    current = ""

    for char in text.lower():
        if char.isalnum() or char == "_":
            current += char
        else:
            if current:
                tokens.append(current)
                current = ""

            tokens.append(char)

    if current:
        tokens.append(current)

    return tokens

In [7]:
# unique words in text in tiny shakespeare
# words2 = sorted(set([' ', "null"] + text.lower().split()))
words = sorted(set(['<UNK>'] + tokenize(text)))
vocab_size = len(words)
print(vocab_size)

11469


In [11]:
# mapping word string to int and reverse
stoi =  { wd:i for i,wd in enumerate(words) }
itos =  { i:wd for i,wd in enumerate(words) }

def encode(s: str):
    words = tokenize(s)
    return [stoi.get(w, stoi['<UNK>']) for w in words]

def decode(li: list):
    return ''.join([itos[i] for i in li])

print(encode('hello there'))
print(decode([0,4,5,6,7]))
print(decode(encode('how are you?')))

[12, 1, 10001]

&',-
how are you?


In [13]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([472819]) torch.int64
tensor([ 3823,     1,  1762,    10,     0,   844,     1, 11064,     1,  7622,
            1,   416,     1,  4180,     6,     1,  4719,     1,  6153,     1,
         9243,     8,     0,     0,   294,    10,     0,  9243,     6,     1,
         9243,     8,     0,     0,  3823,     1,  1762,    10,     0, 11451,
            1,   490,     1,   294,     1,  8204,     1,  7938,     1, 10156,
            1,  2779,     1,  9978,     1, 10156,     1,  3640,    13,     0,
            0,   294,    10,     0,  8204,     8,     1,  8204,     8,     0,
            0,  3823,     1,  1762,    10,     0,  3823,     6,     1, 11451,
            1,  5556,     1,  1431,     1,  6080,     1,  5358,     1,  1697,
            1,  3331,     1, 10156,     1,  9989,     1,  7154,     8,     0])


In [14]:
n = int(0.8*len(data))
#train val split
train_data = data[:n]
val_data = data[n:]


In [ ]:
batch_size = 16
train_data[:16+1] # x of 16 token input giving y which is 17th token output

tensor([ 3823,     1,  1762,    10,     0,   844,     1, 11064,     1,  7622,
            1,   416,     1,  4180,     6,     1,  4719])

In [ ]:
#chunking
torch.manual_seed(8008135)
block_size = 16 # max context length for predictions
batch_size = 6 # how many independent sequence we process in parallel

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data)-block_size, size=(batch_size, )) # get batch_size nums randomly between 0 to len - block_size
    x = torch.stack([data[i:i+block_size] for i in ix])     # input char from rand index to 16 ahead
    y = torch.stack([data[i+1:i+1+block_size] for i in ix]) # respective output for that same tensor matrix index like 0,0 in x input gives 0,0 y index value.
    return x,y
    
xb, yb = get_batch('train')     # input & output for the model
print('input batch x: ', xb.shape)
print(xb)
print('output batch y: ', yb.shape)
print(yb)

print('---representation---')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {decode(context.tolist())!r} the target: {decode([target.item()])!r}")

input batch x:  torch.Size([6, 16])
tensor([[    6,     1, 11167,     1,  9559,     1, 10730,     1,  6545,     1,
          4730,     6,     0, 11167,     1,  5358],
        [    7,     7,  5742,     1,  6545,     1,  7694,     0,  1893,     1,
          4907,     1, 10156,     1, 11426,     2],
        [    6,     1, 10015,     1,  3943,     1, 10744,     1, 11282,     1,
         11251,    11,     0,   378,     1, 11065],
        [  650,     6,     1,  9987,     1,   797,     1,  1183,     1,  6148,
             1,  5842,    10,     0,  4573,     1],
        [    1, 11451,     1,  1461,     1,  2996,     1, 10845,     1,  5841,
             1,   313,    11,     1,  3969,     1],
        [    0,   962,     1,  9989,     1,  8353,    11,     1,  9989,     1,
         10117,     1, 11229,     1, 11032,     1]])
output batch y:  torch.Size([6, 16])
tensor([[    1, 11167,     1,  9559,     1, 10730,     1,  6545,     1,  4730,
             6,     0, 11167,     1,  5358,     1],
        [

### Neural Network part begins yay

In [28]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(8008135)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets):  # idx -> input token IDs, targets -> output token IDs
        logits = self.token_embedding_table(idx) # (B, T, C) -> B = batch size, T = time steps (sequence length like 16 batch size in training), C = vocab_size (channels)
        # logits are raw prediction scores before converting into probabilities
        B, T, C = logits.shape
        logits = logits.view(B*T, C)  # reshape logits to (B*T, C) for loss computation (pytorch expects 2D input for cross_entropy)
        targets = targets.view(B*T)  # 1D tensor
        loss = F.cross_entropy(logits, targets)  # compute the loss between predicted logits and actual targets, lower is better
        return logits, loss

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([96, 11469])
tensor(9.7698, grad_fn=<NllLossBackward0>)
